In [2]:
import os
import random
import csv
from pathlib import Path

from PIL import Image
import torchvision.transforms as T
from torchvision.transforms.functional import to_pil_image

# Config
input_root = Path(".")
output_root = Path("../preprocessed/Dataset_BUSI_with_GT")
csv_path = output_root / "all_captions.csv"

classes = ["benign", "malignant", "normal"]
tissue = "breast"

random.seed(42)

# Prompt templates
templates = [
    "An ultrasound image of {Tissue} with {Condition} findings.",
    "A B-mode ultrasound of {Tissue}, consistent with {Condition}.",
    "Sonographic appearance of {Condition} {Tissue}.",
    "This ultrasound demonstrates {Tissue} exhibiting features of {Condition}.",
    "A clinical ultrasound scan of {Tissue}, indicative of {Condition} pathology.",
    "Echographic findings of {Tissue} showing {Condition} characteristics.",
    "This sonogram of {Tissue} is consistent with a {Condition} diagnosis.",
    "A diagnostic ultrasound image of {Tissue}, with imaging features suggestive of {Condition}.",
    "Ultrasound of {Tissue} presenting sonographic signs of {Condition}.",
    "A grayscale ultrasound demonstrating {Condition} changes in {Tissue}.",
]

# Transform
transform = T.Compose([
    T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
    T.Resize((256, 256)),
    T.ToTensor(),
])

# Helper functions
def is_valid_image(p):
    return p.suffix.lower() in [".png", ".jpg", ".jpeg"]

def extract_index(path: Path):
    try:
        return int(path.stem.split("(")[-1].replace(")", ""))
    except:
        return float("inf")

def generate_caption(tissue, condition, name):
    idx = hash(name) % len(templates)   # deterministic
    template = templates[idx]
    return template.format(Tissue=tissue, Condition=condition)

# Main
rows = []
output_root.mkdir(parents=True, exist_ok=True)

for cls in classes:
    input_dir = input_root / cls
    output_dir = output_root / cls
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nProcessing {cls}...")

    image_files = sorted(
        [
            p for p in input_dir.iterdir()
            if p.is_file()
            and is_valid_image(p)
            and "mask" not in p.name.lower()
        ],
        key=extract_index
    )

    for img_path in image_files:
        try:
            img = Image.open(img_path)
            img_tensor = transform(img)

            # save image
            save_path = output_dir / img_path.name
            to_pil_image(img_tensor).save(save_path)

            # generate caption
            caption = generate_caption(tissue, cls, img_path.name)

            # record (include relative path for training later)
            rows.append({
                "image_path": f"{cls}/{img_path.name}",
                "text_caption": caption
            })

        except Exception as e:
            print(f"Failed on {img_path.name}: {e}")

# Save CSV
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["image_path", "text_caption"])
    writer.writeheader()
    writer.writerows(rows)

print(f"\nDone. Total images: {len(rows)}")
print(f"CSV saved to: {csv_path}")


Processing benign...

Processing malignant...

Processing normal...

Done. Total images: 780
CSV saved to: ../preprocessed/Dataset_BUSI_with_GT/all_captions.csv


In [6]:
import csv
from pathlib import Path

# =========================
# Config
# =========================
data_root = Path("/Users/guest/Coding/Harvard/AIM2/project/BMI702_project/dataset/preprocessed/Dataset_BUSI_with_GT")
csv_path = data_root / "all_captions.csv"

classes = ["benign", "malignant", "normal"]
tissue = "breast"

random.seed(42)

# =========================
# Prompt templates
# =========================
templates = [
    "An ultrasound image of {Tissue} with {Condition} findings.",
    "A B-mode ultrasound of {Tissue}, consistent with {Condition}.",
    "Sonographic appearance of {Condition} {Tissue}.",
    "This ultrasound demonstrates {Tissue} exhibiting features of {Condition}.",
    "A clinical ultrasound scan of {Tissue}, indicative of {Condition} pathology.",
    "Echographic findings of {Tissue} showing {Condition} characteristics.",
    "This sonogram of {Tissue} is consistent with a {Condition} diagnosis.",
    "A diagnostic ultrasound image of {Tissue}, with imaging features suggestive of {Condition}.",
    "Ultrasound of {Tissue} presenting sonographic signs of {Condition}.",
    "A grayscale ultrasound demonstrating {Condition} changes in {Tissue}.",
]

def generate_caption(tissue, condition, name):
    idx = hash(name) % len(templates)
    template = templates[idx]
    return template.format(Tissue=tissue, Condition=condition)

# =========================
# Build CSV
# =========================
rows = []

for cls in classes:
    class_dir = data_root / cls

    if not class_dir.exists():
        print(f"Warning: {class_dir} does not exist")
        continue

    for img_path in sorted(class_dir.iterdir()):
        if not img_path.is_file():
            continue

        if img_path.suffix.lower() not in [".png", ".jpg", ".jpeg"]:
            continue

        if "mask" in img_path.name.lower():
            continue

        caption = generate_caption(tissue, cls, img_path.name)

        rows.append({
            "image_path": f"{cls}/{img_path.name}",
            "text_caption": caption,
            "label": cls   # ✅ new column
        })

# =========================
# Save CSV
# =========================
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["image_path", "text_caption", "label"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Done. Total images: {len(rows)}")
print(f"CSV saved to: {csv_path}")

Done. Total images: 780
CSV saved to: /Users/guest/Coding/Harvard/AIM2/project/BMI702_project/dataset/preprocessed/Dataset_BUSI_with_GT/all_captions.csv
